In [16]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA, KernelPCA
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score


In [17]:
# The notebook kernel may start outside the project folder.
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
candidates += [folder / "applied-ml-midterm" for folder in candidates.copy()]
# Local project location, used if the kernel starts in an unrelated folder.
candidates.append(Path("/Users/prith/Documents/Cornell/Applied Machine Learning/midterm/applied-ml-midterm"))

PROJECT_DIR = next((folder for folder in candidates if (folder / "part1-data").is_dir()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError(f"Could not find part1-data. Kernel working directory: {cwd}")

ATTEMPT_DIR = PROJECT_DIR / "prith_attempts" / "part_1"
DATA_DIR = PROJECT_DIR / "part1-data"
print("Loading data from:", DATA_DIR)

part1_train = pd.read_csv(DATA_DIR / "part1_train.csv")
part1_val = pd.read_csv(DATA_DIR / "part1_val.csv")
part1_test = pd.read_csv(DATA_DIR / "part1_test.csv")
pathway_catalog = pd.read_csv(DATA_DIR / "pathway_catalog.csv")

print("Training:", part1_train.shape)
print("Validation:", part1_val.shape)
print("Test:", part1_test.shape)
print("Pathway catalog:", pathway_catalog.shape)

part1_train.head()


Loading data from: /Users/prith/Documents/Cornell/Applied Machine Learning/midterm/applied-ml-midterm/part1-data
Training: (1171, 166)
Validation: (78, 166)
Test: (312, 161)
Pathway catalog: (2501, 5)


,ID,start_year,criteria,diseases_cleaned,drugs,icdcodes,matched_drugs,drug_targets,drug_actions,drug_pathways,...,emb_145,emb_146,emb_147,emb_148,emb_149,Safety,Funding,Efficacy,Recruitment,Other
0,TR08860,2003,\n Inclusion Criteria:\n\n - ...,"['lung cancer, non-small cell']",['gw572016 (lapatinib)'],"[""['C78.00', 'C78.01', 'C78.02', 'D14.30', 'D1...",NaN,NaN,NaN,NaN,...,0.057633,-0.018126,-0.130373,-0.072231,-0.051629,-100,-100,-100,-100,-100
1,TR03494,2004,\n DISEASE CHARACTERISTICS:\n\n ...,['leukemia'],"['autologous tumor cell vaccine', 'therapeutic...","[""['C95.91', 'C95.92', 'Z80.6', 'Z85.6', 'C90....",NaN,NaN,NaN,NaN,...,-0.018860,-0.077071,-0.059479,0.009044,0.021266,-100,-100,-100,-100,-100
2,TR00772,2006,\n Inclusion Criteria:\n\n - ...,"['clear cell renal cell carcinoma', 'recurrent...",['cediranib maleate'],"[""['C22.0', 'C4A.9', 'C7B.1', 'C4A.0', 'C4A.31...",NaN,NaN,NaN,NaN,...,-0.019414,-0.007497,-0.062808,-0.071619,-0.078172,-100,-100,-100,-100,-100
3,TR08870,2007,\n Inclusion Criteria:\n\n - ...,"['pulmonary disease, chronic obstructive']","['uk-432,097', 'placebo']","[""['J44.9', 'J44.1', 'J44.0']""]",NaN,NaN,NaN,NaN,...,0.014069,-0.006230,-0.081606,-0.031338,-0.063282,-100,-100,-100,-100,-100
4,TR02827,2004,\n Inclusion Criteria:\n\n - ...,"['schizophrenia', 'schizoaffective disorder']",['ziprasidone'],"[""['F20.0', 'F20.1', 'F20.2', 'F20.3', 'F20.5'...",ziprasidone,ADRA1A;ADRA1B;ADRA2A;ADRA2B;ADRA2C;CHRM1;CHRM2...,agonist;antagonist;binder,R-HSA-109582;R-HSA-112314;R-HSA-112315;R-HSA-1...,...,0.093187,-0.043244,-0.077393,-0.110769,0.016161,-100,-100,-100,-100,-100


In [18]:
def create_scaled_embeddings(train_df, val_df, test_df):
    embedding_columns = [f"emb_{i}" for i in range(150)]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_df[embedding_columns])
    X_val = scaler.transform(val_df[embedding_columns])
    X_test = scaler.transform(test_df[embedding_columns])

    return X_train, X_val, X_test, scaler



In [19]:
def inspect_cluster_percentages(model, X_train, train_df, percentage_threshold):
    reason_columns = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
    cluster_ids = model.predict(X_train)
    labeled_mask = train_df[reason_columns].isin([0, 1]).all(axis=1).to_numpy()

    clusters = range(model.n_components)
    cluster_reason_mapping = {}
    for cluster in clusters:
        labeled_positions = np.flatnonzero((cluster_ids == cluster) & labeled_mask)
        cluster_reason_mapping[int(cluster)] = []
        if len(labeled_positions) == 0:
            continue

        labeled_rows = train_df.iloc[labeled_positions]
        percentages = labeled_rows[reason_columns].mean() * 100
        cluster_reason_mapping[int(cluster)] = [
            reason for reason in reason_columns
            if percentages[reason] >= percentage_threshold
        ]

    return cluster_reason_mapping


In [20]:
def label_unlabeled_points(model, X_train, train_df, cluster_reason_mapping):
    reason_columns = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
    known_mask = train_df[reason_columns].isin([0, 1]).all(axis=1).to_numpy()
    unlabeled_mask = train_df[reason_columns].eq(-100).all(axis=1).to_numpy()

    result = train_df.copy()
    result["label_source"] = np.where(known_mask, "observed", "unresolved").astype(object)
    cluster_ids = model.predict(X_train)
    reason_positions = result.columns.get_indexer(reason_columns)

    for position in np.flatnonzero(unlabeled_mask):
        reasons = cluster_reason_mapping.get(int(cluster_ids[position]), [])
        if not reasons:
            continue
        result.iloc[position, reason_positions] = [
            int(reason in reasons) for reason in reason_columns
        ]
        result.iloc[position, result.columns.get_loc("label_source")] = "cluster_prediction"

    return result


In [21]:
def train_logistic_regression(X_train, train_df, C=1.0, max_iter=500,
                              random_state=42):
    """Train five classifiers using observed and recovered labels equally."""
    reason_columns = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
    labeled_mask = train_df[reason_columns].isin([0, 1]).all(axis=1).to_numpy()
    if not labeled_mask.any():
        raise ValueError("No rows have five known binary reason labels for training.")

    y_train = train_df.loc[labeled_mask, reason_columns].to_numpy(dtype=int)
    model = OneVsRestClassifier(
        LogisticRegression(
            C=C,
            max_iter=max_iter,
            solver="liblinear",
            random_state=random_state,
        )
    )
    model.fit(X_train[labeled_mask], y_train)
    return model


In [22]:
def predict_reason_labels(model, X):
    """Predict binary reasons, choosing the highest-probability reason if all are zero."""
    predictions = np.asarray(model.predict(X)).copy()
    empty_rows = ~predictions.any(axis=1)
    if empty_rows.any():
        probabilities = model.predict_proba(X[empty_rows])
        best_reasons = np.asarray(probabilities).argmax(axis=1)
        predictions[np.flatnonzero(empty_rows), best_reasons] = 1
    return predictions


In [23]:
def evaluate_reason_model(model, X_val, val_df):
    reason_columns = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
    y_true = val_df[reason_columns].to_numpy(dtype=int)
    y_pred = predict_reason_labels(model, X_val)
    per_reason_f1 = f1_score(y_true, y_pred, average=None, zero_division=0)

    return pd.Series(
        [
            f1_score(y_true, y_pred, average="samples", zero_division=0),
            f1_score(y_true, y_pred, average="macro", zero_division=0),
            *per_reason_f1,
        ],
        index=["Sample-wise F1", "Macro F1", *reason_columns],
        name="Validation F1",
    )


In [24]:
def create_pca_embeddings(X_train, n_components=50):
    """Fit PCA on scaled training embeddings and return embeddings plus fitted PCA."""
    pca = PCA(n_components=n_components, svd_solver="full")
    X_train_pca = pca.fit_transform(X_train)
    return X_train_pca, pca


In [25]:
def create_kernel_pca_embeddings(X_train, n_components=50, kernel="rbf",
                                 gamma=None, random_state=42):
    """Fit kernel PCA on scaled training embeddings; return embeddings and transformer."""
    effective_gamma = 1 / X_train.shape[1] if gamma is None else gamma
    kernel_pca = KernelPCA(
        n_components=n_components,
        kernel=kernel,
        gamma=effective_gamma,
        eigen_solver="arpack",
        random_state=random_state,
    )
    X_train_kernel_pca = kernel_pca.fit_transform(X_train)
    return X_train_kernel_pca, kernel_pca


In [26]:
def train_gmm(X_train, n_components, covariance_type="diag", n_init=5,
              max_iter=300, reg_covar=1e-4, random_state=42):
    """Fit a Gaussian mixture with EM and return the fitted clustering model."""
    model = GaussianMixture(
        n_components=n_components,
        covariance_type=covariance_type,
        n_init=n_init,
        max_iter=max_iter,
        reg_covar=reg_covar,
        random_state=random_state,
    )
    model.fit(X_train)
    return model


In [27]:
def label_unlabeled_points_soft(model, X_train, train_df, percentage_threshold=50):
    """Recover labels using soft GMM membership; return a copy and inspection details.

    Details contain per-trial reason probabilities, component reason rates, and
    effective labeled support. All rates are 0-1; the threshold is a percentage.
    Only original known labels estimate rates; empty predictions stay unresolved.
    """
    reason_columns = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
    if len(X_train) != len(train_df):
        raise ValueError("Embeddings and trial rows must have matching lengths and order.")
    if not 0 <= percentage_threshold <= 100:
        raise ValueError("percentage_threshold must be between 0 and 100.")

    known_mask = train_df[reason_columns].isin([0, 1]).all(axis=1).to_numpy()
    unlabeled_mask = train_df[reason_columns].eq(-100).all(axis=1).to_numpy()
    if not (known_mask | unlabeled_mask).all():
        raise ValueError("Each trial must have five known labels or five masked labels.")
    if not known_mask.any():
        raise ValueError("At least one originally labeled trial is needed to estimate reason rates.")

    responsibilities = np.asarray(model.predict_proba(X_train), dtype=float)
    known_labels = train_df.loc[known_mask, reason_columns].to_numpy(dtype=float)
    known_responsibilities = responsibilities[known_mask]
    effective_support = known_responsibilities.sum(axis=0)

    # Each known trial contributes its membership weight to every component.
    weighted_positive_counts = known_responsibilities.T @ known_labels
    overall_reason_rates = known_labels.mean(axis=0)
    component_reason_rates = np.tile(overall_reason_rates, (responsibilities.shape[1], 1))
    # Unsupported components fall back to overall known-training reason rates.
    np.divide(
        weighted_positive_counts, effective_support[:, None],
        out=component_reason_rates, where=effective_support[:, None] > 1e-12,
    )

    # Mix component reason rates using each trial's own membership probabilities.
    reason_probabilities = np.clip(responsibilities @ component_reason_rates, 0, 1)
    predictions = (reason_probabilities >= percentage_threshold / 100).astype(int)
    assign_mask = unlabeled_mask & predictions.any(axis=1)

    result = train_df.copy()
    result["label_source"] = np.where(known_mask, "observed", "unresolved").astype(object)
    reason_positions = result.columns.get_indexer(reason_columns)
    assigned_positions = np.flatnonzero(assign_mask)
    result.iloc[assigned_positions, reason_positions] = predictions[assign_mask]
    result.iloc[assigned_positions, result.columns.get_loc("label_source")] = "probability_prediction"

    soft_details = {
        "reason_probabilities": pd.DataFrame(
            reason_probabilities, index=train_df.index, columns=reason_columns
        ),
        "component_reason_rates": pd.DataFrame(
            component_reason_rates, columns=reason_columns
        ).rename_axis("component"),
        "effective_labeled_support": pd.Series(
            effective_support, name="Effective labeled support"
        ).rename_axis("component"),
    }
    return result, soft_details


In [28]:
# Run the imports, data-loading, and function-definition cells first.
PERCENTAGE_THRESHOLD = 50
RANDOM_STATE = 42  # Kernel PCA and logistic regression stay fixed.
GMM_RANDOM_STATE = 42
GMM_N_INIT = 5
GMM_COVARIANCE_TYPE = "diag"
GMM_REG_COVAR = 1e-4
GMM_MAX_ITER = 300
KERNEL_PCA_COMPONENTS = 50
KERNEL = "rbf"
KERNEL_GAMMA = 1 / 75

# Scaling is fitted on training embeddings only.
X_train, X_val, X_test, scaler = create_scaled_embeddings(
    part1_train, part1_val, part1_test
)

# Kernel PCA is used only for clustering and label recovery.
X_train_cluster, kernel_pca_model = create_kernel_pca_embeddings(
    X_train, n_components=KERNEL_PCA_COMPONENTS, kernel=KERNEL,
    gamma=KERNEL_GAMMA, random_state=RANDOM_STATE,
)
print(f"Clustering uses {X_train_cluster.shape[1]} kernel PCA components.")
print(f"Kernel: {kernel_pca_model.kernel}; gamma: {kernel_pca_model.gamma:.6g}")

# The baseline and augmented classifiers both use the original scaled embeddings.
baseline_model = train_logistic_regression(
    X_train, part1_train, C=1.0, max_iter=500, random_state=RANDOM_STATE,
)
baseline_validation_metrics = evaluate_reason_model(baseline_model, X_val, part1_val)

pipeline_results = {}
# Compare mixture component counts from 2 through 30, inclusive.
for n_components in range(2, 31):
    gmm_model = train_gmm(
        X_train_cluster, n_components=n_components,
        covariance_type=GMM_COVARIANCE_TYPE, n_init=GMM_N_INIT,
        max_iter=GMM_MAX_ITER, reg_covar=GMM_REG_COVAR,
        random_state=GMM_RANDOM_STATE,
    )
    print(f"Components: {n_components}; converged: {gmm_model.converged_}; iterations: {gmm_model.n_iter_}")
    # Soft memberships determine each trial's own five reason probabilities.
    part1_train_augmented, soft_label_details = label_unlabeled_points_soft(
        gmm_model, X_train_cluster, part1_train,
        percentage_threshold=PERCENTAGE_THRESHOLD,
    )
    reason_model = train_logistic_regression(
        X_train, part1_train_augmented,
        C=1.0, max_iter=500, random_state=RANDOM_STATE,
    )
    validation_metrics = evaluate_reason_model(reason_model, X_val, part1_val)

    pipeline_results[n_components] = {
        "gmm_model": gmm_model,
        "gmm_converged": bool(gmm_model.converged_),
        "gmm_iterations": int(gmm_model.n_iter_),
        "gmm_settings": {
            "n_components": n_components,
            "covariance_type": GMM_COVARIANCE_TYPE,
            "n_init": GMM_N_INIT,
            "max_iter": GMM_MAX_ITER,
            "reg_covar": GMM_REG_COVAR,
            "random_state": GMM_RANDOM_STATE,
        },
        "kernel_pca_model": kernel_pca_model,
        "labeling_method": "soft_gmm_responsibilities",
        "soft_reason_probabilities": soft_label_details["reason_probabilities"],
        "cluster_soft_reason_rates": soft_label_details["component_reason_rates"],
        "effective_labeled_support": soft_label_details["effective_labeled_support"],
        "part1_train_augmented": part1_train_augmented,
        "reason_model": reason_model,
        "validation_metrics": validation_metrics,
        "label_source_counts": part1_train_augmented["label_source"].value_counts(),
    }


Clustering uses 50 kernel PCA components.
Kernel: rbf; gamma: 0.0133333
Components: 2; converged: True; iterations: 13
Components: 3; converged: True; iterations: 9
Components: 4; converged: True; iterations: 25
Components: 5; converged: True; iterations: 18
Components: 6; converged: True; iterations: 16
Components: 7; converged: True; iterations: 28
Components: 8; converged: True; iterations: 19
Components: 9; converged: True; iterations: 23
Components: 10; converged: True; iterations: 22
Components: 11; converged: True; iterations: 62
Components: 12; converged: True; iterations: 25
Components: 13; converged: True; iterations: 19
Components: 14; converged: True; iterations: 25
Components: 15; converged: True; iterations: 22
Components: 16; converged: True; iterations: 20
Components: 17; converged: True; iterations: 21
Components: 18; converged: True; iterations: 15
Components: 19; converged: True; iterations: 22
Components: 20; converged: True; iterations: 22
Components: 21; converged

In [29]:
validation_scores = pd.DataFrame.from_dict(
    {
        n_components: run["validation_metrics"]
        for n_components, run in pipeline_results.items()
    },
    orient="index",
).sort_index()
validation_scores.index.name = "Number of components"
validation_scores = validation_scores[
    ["Sample-wise F1", "Macro F1", "Safety", "Funding", "Efficacy", "Recruitment", "Other"]
]
# Percentage of all training trials still unlabeled after recovery.
reason_columns = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
validation_scores["Percent unlabeled"] = pd.Series({
    n_components: (
        run["part1_train_augmented"][reason_columns].eq(-100).all(axis=1).mean() * 100
    )
    for n_components, run in pipeline_results.items()
})
validation_scores["Delta vs baseline"] = (
    validation_scores["Sample-wise F1"] - baseline_validation_metrics["Sample-wise F1"]
)
# Inspect convergence as well as predictive performance for every component count.
validation_scores["GMM converged"] = pd.Series({
    n_components: run["gmm_converged"]
    for n_components, run in pipeline_results.items()
})
validation_scores["GMM iterations"] = pd.Series({
    n_components: run["gmm_iterations"]
    for n_components, run in pipeline_results.items()
})
# Soft support is the summed known-trial membership mass, not an integer count.
validation_scores["Min soft labeled support"] = pd.Series({
    n_components: run["effective_labeled_support"].min()
    for n_components, run in pipeline_results.items()
})
print(f"Observed-label-only baseline sample-wise F1: {baseline_validation_metrics['Sample-wise F1']:.4f}")
with pd.option_context("display.max_columns", None, "display.max_rows", None):
    display(validation_scores.round(4))

# Select using the original scores, not their rounded display values.
samplewise_f1_scores = validation_scores["Sample-wise F1"]
best_samplewise_f1 = samplewise_f1_scores.max()
best_component_counts = samplewise_f1_scores.index[
    samplewise_f1_scores == best_samplewise_f1
].tolist()
print("Best component count(s):", best_component_counts)
print(f"Highest validation sample-wise F1: {best_samplewise_f1:.4f}")
# Component-count selection on only 78 validation trials can overfit validation.
# Compare the selected result with the supervised baseline in the report.


Observed-label-only baseline sample-wise F1: 0.8782


,Sample-wise F1,Macro F1,Safety,Funding,Efficacy,Recruitment,Other,Percent unlabeled,Delta vs baseline,GMM converged,GMM iterations,Min soft labeled support
Number of components,,,,,,,,,,,,
2,0.6496,0.2351,0.2000,0.0000,0.0000,0.7937,0.1818,0.0000,-0.2286,True,13,67.9126
3,0.7308,0.5033,0.6154,0.4000,0.3529,0.8403,0.3077,31.5969,-0.1474,True,9,23.3339
4,0.7350,0.4853,0.5333,0.4000,0.3529,0.8547,0.2857,26.7293,-0.1432,True,25,24.0000
5,0.7714,0.5696,0.6667,0.3333,0.4444,0.8772,0.5263,36.0376,-0.1068,True,18,7.0000
6,0.7799,0.5821,0.6250,0.3333,0.5000,0.8807,0.5714,34.2442,-0.0983,True,16,2.0000
7,0.8581,0.6727,0.7619,0.2857,0.7857,0.9703,0.5600,51.0675,-0.0201,True,28,0.0000
8,0.7774,0.5786,0.6667,0.3333,0.4444,0.8772,0.5714,33.3049,-0.1009,True,19,0.0000
9,0.7628,0.5502,0.5714,0.3333,0.4211,0.8696,0.5556,30.1452,-0.1154,True,23,0.0000
10,0.7735,0.5734,0.7143,0.3333,0.5000,0.8750,0.4444,28.6934,-0.1047,True,22,0.0000


Best component count(s): [7]
Highest validation sample-wise F1: 0.8581


In [30]:
def export_part1_predictions(model, X_test, test_df, output_path):
    """Save Kaggle ID,Predicted rows with at least one reason per trial."""
    reason_columns = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
    if len(X_test) != len(test_df):
        raise ValueError("Test embeddings and trial rows must have matching lengths and order.")
    predictions = predict_reason_labels(model, X_test)
    if predictions.shape != (len(test_df), len(reason_columns)):
        raise ValueError("Expected one five-label prediction vector per test trial.")
    if not np.isin(predictions, [0, 1]).all():
        raise ValueError("Reason predictions must be binary.")
    if test_df["ID"].isna().any() or not test_df["ID"].is_unique:
        raise ValueError("Test trial IDs must be present and unique.")

    submission = test_df[["ID"]].copy()
    submission["Predicted"] = [
        " ".join(reason for reason, value in zip(reason_columns, row) if value == 1)
        for row in predictions
    ]
    submission.to_csv(output_path, index=False)
    return submission


In [31]:
# Choose a run explicitly after reviewing all metrics and unresolved trials.
# Change this value to export another component count; never use the loop's last model.
GMM_EXPORT_COMPONENTS = 21
if GMM_EXPORT_COMPONENTS not in pipeline_results:
    raise ValueError("Run the component sweep before exporting the selected model.")
selected_run = pipeline_results[GMM_EXPORT_COMPONENTS]
if selected_run.get("labeling_method") != "soft_gmm_responsibilities":
    raise ValueError("Run the soft-membership pipeline before exporting its model.")
# Kernel PCA recovered training labels; the classifier uses original scaled embeddings.
selected_reason_model = selected_run["reason_model"]
submission_path = ATTEMPT_DIR / "part1_gmm_kernel_pca_submission.csv"
part1_submission = export_part1_predictions(
    selected_reason_model, X_test, part1_test, submission_path
)
print("Saved:", submission_path)
print("Test trials:", len(part1_submission))
print("Trials with no predicted reasons:", part1_submission["Predicted"].eq("").sum())
print("Highest-probability fallback trials:", (~selected_reason_model.predict(X_test).any(axis=1)).sum())
part1_submission.head()


Saved: /Users/prith/Documents/Cornell/Applied Machine Learning/midterm/applied-ml-midterm/part1_gmm_kernel_pca_submission.csv
Test trials: 312
Trials with no predicted reasons: 0
Highest-probability fallback trials: 23


,ID,Predicted
0,TR00756,Safety Efficacy Other
1,TR05267,Safety Other
2,TR10354,Safety
3,TR01930,Recruitment
4,TR01046,Safety
